# Создание моделей для предсказания свойств углепластика, полученного по вакуумной технологии

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import f_classif

import matplotlib.pyplot as plt 

# инструменты для построения модели:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # инструмент для создания и обучения модели
from sklearn.ensemble import RandomForestRegressor # инструмент для создания и обучения модели
from sklearn import metrics # инструменты для оценки точности модели
from xgboost import XGBRegressor

import pickle

RANDOM_SEED = 42


In [2]:
df = pd.read_csv('data/dataset_prepaired.csv')
df.head()

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,technology,Thickness of the monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
0,188.0,1.758,4.59,253.0,1.814229,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
1,189.0,1.758,4.48,260.0,1.723077,1.1,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
2,188.0,1.759,4.28,257.0,1.665370,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
3,187.0,1.758,4.77,256.0,1.863281,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
4,190.0,1.757,4.56,255.0,1.788235,0.9,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0


In [3]:

df = df[df['technology'] == 1]
df = df.drop('technology', axis=1)

In [4]:
df = df.rename(columns={'Thickness of the monolayer' : 'Thickness_monolayer',
                        'Module_plastik ' : 'Module_plastik'})

In [5]:
X = np.array([[190, 1.779, 4.6, 265, 1.7, 1.3, 20, 210, 333, 37.40, 32.33, 12.8, 149.0]])

### Модель для предсказания толщины монослоя Thickness_monolayer

In [6]:
train_data_thickness = df.drop(['density', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_thickness.drop(['Thickness_monolayer'], axis=1))
y = np.array(train_data_thickness.Thickness_monolayer.values)

In [7]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [8]:
# НАСТРОЙКИ 
model_rf_thikness = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

In [9]:
# обучаем модель на тестовом наборе данных
model_rf_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_thikness.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [10]:
def mean_absolute_percentage_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr) / y_tr)) * 100

In [11]:
# сравниваем предсказанные значения (y_pred) с реальными (y_test), 
# метрика mean squared error, MSE показывает среднеквадратичное отклонение:

def mean_squared_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr)**2)) 

In [12]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.011


In [13]:
# save model
with open('autoclave_model_rf_thikness.pkl','wb') as f:
    pickle.dump(model_rf_thikness,f)

In [14]:
model_lr_thikness = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_thikness.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 1.389


In [15]:
# save model
with open('autoclave_model_lr_thikness.pkl','wb') as f:
    pickle.dump(model_lr_thikness,f)

In [16]:
print('Линейная регрессия. Толщина монослоя:', model_lr_thikness.predict(X))
print('Случайный лес регрессия. Толщина монослоя:', model_rf_thikness.predict(X))

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


Линейная регрессия. Толщина монослоя: [0.20621994 0.20795276 0.20682424 ... 0.21377687 0.21341728 0.21372296]
Случайный лес регрессия. Толщина монослоя: [0.209      0.209      0.209      ... 0.21366667 0.21366667 0.21366667]


### Модель для предсказания плотности углепластика density

In [17]:
train_data_density = df.drop(['Thickness_monolayer', 'Strength_plastik', 'Module_plastik',
                            'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_density.drop(['density'], axis=1))
y = np.array(train_data_density.density.values)

In [18]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_density = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_density.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [19]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.001


In [20]:
# save model
with open('autoclave_model_rf_density.pkl','wb') as f:
    pickle.dump(model_rf_density,f)

In [21]:
model_lr_density = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_density.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.404


In [22]:
# save model
with open('autoclave_model_lr_density.pkl','wb') as f:
    pickle.dump(model_lr_density,f)

In [23]:
print('Линейная регрессия. Плотность:', model_lr_density.predict(X))
print('Случайный лес регрессия. Плотность:', model_rf_density.predict(X))

Линейная регрессия. Плотность: [1.54256972 1.54312033 1.5451962  ... 1.53459176 1.53438561 1.532231  ]
Случайный лес регрессия. Плотность: [1.555      1.555      1.555      ... 1.52566667 1.52566667 1.52566667]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


### Модель для предсказания прочности углепластика Strength

In [24]:
df['Strength Gpa'] = df['Strength Gpa'] * 1000
df

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,Thickness_monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
426,188.0,1.752,4290.0,254.0,1.688976,1.100000,26.20,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
427,184.0,1.751,4300.0,255.0,1.686275,1.200000,24.70,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
428,179.0,1.750,4620.0,268.0,1.723881,1.200000,26.30,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
429,189.0,1.749,4370.0,254.0,1.720472,1.100000,25.60,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
430,189.0,1.749,4260.0,255.0,1.670588,1.000000,24.80,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10706,183.0,1.763,4750.0,263.0,1.806084,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0
10707,183.0,1.764,4610.0,262.0,1.759542,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0
10708,186.0,1.763,4440.0,265.0,1.675472,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0
10709,184.0,1.763,4450.0,257.0,1.731518,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0


In [25]:
train_data_strength = df.drop(['Thickness_monolayer', 'Module_plastik',
       'LSS', 'Plastik_Tg', 'density'], axis=1)

X = np.array(train_data_strength.drop(['Strength_plastik'], axis=1))
y = np.array(train_data_strength.Strength_plastik.values)

In [26]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [27]:
# НАСТРОЙКИ 
model_rf_strength = RandomForestRegressor(
    n_estimators=500, 
    max_features=4,
    min_samples_leaf=6,
    max_depth=8,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_strength.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    0.4s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


In [28]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 11.886
MAPE: 0.041


In [29]:
# save model
with open('autoclave_model_rf_strength.pkl','wb') as f:
    pickle.dump(model_rf_strength,f)

In [30]:
model_lr_strength = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_strength.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 479.289
MAPE: 1.833


In [31]:
# save model
with open('autoclave_model_lr_strength.pkl','wb') as f:
    pickle.dump(model_lr_strength,f)

In [34]:
model_xgb_strenght = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_strenght.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_strenght.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

/home/alexandr/anaconda3/envs/ML/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:52:04] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MSE: 21.984
MAPE: 0.032


In [35]:
model_xgb_strenght.predict(X)

array([921.9841 , 922.00104, 922.01154, ..., 871.9993 , 872.0026 ,
       872.00226], shape=(6913,), dtype=float32)

In [36]:
# save model
with open('autoclave_model_xgb_strenght.pkl','wb') as f:
    pickle.dump(model_xgb_strenght,f)

In [37]:
print('Линейная регрессия. Прочность:', model_lr_strength.predict(X))
print('Случайный лес регрессия. Прочность:', model_rf_strength.predict(X))
print('XGB регрессия. Прочность:', model_xgb_strenght.predict(X))

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.1s


Линейная регрессия. Прочность: [909.98103455 914.01660472 914.49189484 ... 907.70671071 911.05852282
 905.72681227]
Случайный лес регрессия. Прочность: [921.81005828 921.9820951  922.06723989 ... 872.00100337 872.00058053
 872.00164836]
XGB регрессия. Прочность: [921.9841  922.00104 922.01154 ... 871.9993  872.0026  872.00226]


[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.1s finished


### Модель для предсказания модуля углепластика Module

In [38]:
train_data_module = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_module.drop(['Module_plastik'], axis=1))
y = np.array(train_data_module.Module_plastik .values)

In [39]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_module = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_module.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [40]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.018
MAPE: 0.011


In [41]:
# save model
with open('autoclave_model_rf_module.pkl','wb') as f:
    pickle.dump(model_rf_module,f)

In [42]:
model_lr_module = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_module.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 2.988
MAPE: 2.183


In [43]:
# save model
with open('autoclave_model_lr_module.pkl','wb') as f:
    pickle.dump(model_lr_module,f)

In [ ]:
print('Линейная регрессия. Модуль упругости:', model_lr_module.predict(X))
print('Случайный лес регрессия. Модуль упругости:', model_rf_module.predict(X))

Линейная регрессия. Модуль упругости: [71.40255516 70.75776035 70.86249121 ... 64.0916589  64.68498746
 64.44990586]
Случайный лес регрессия. Модуль упругости: [71.7  71.7  71.7  ... 65.95 65.95 65.95]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


### Модель для предсказания межслоевой прочности углепластика LSS

In [45]:
train_data_lss = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik',
                             'Plastik_Tg'], axis=1)

X = np.array(train_data_lss.drop(['LSS'], axis=1))
y = np.array(train_data_lss.LSS.values)

In [46]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_lss = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_lss.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [47]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.156
MAPE: 0.032


In [48]:
# save model
with open('autoclave_model_rf_lss.pkl','wb') as f:
    pickle.dump(model_rf_lss,f)

In [49]:
model_lr_lss = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_lss.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 16.151
MAPE: 4.074


In [51]:
# save model
with open('autoclave_model_lr_lss.pkl','wb') as f:
    pickle.dump(model_lr_lss,f)

In [52]:
model_xgb_lss = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_lss.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

/home/alexandr/anaconda3/envs/ML/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:52:05] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MSE: 0.155
MAPE: 0.026


In [53]:
# save model
with open('autoclave_model_xgb_lss.pkl','wb') as f:
    pickle.dump(model_xgb_lss,f)

In [54]:
print('Линейная регрессия. Прочность при сдвиге:', model_lr_lss.predict(X))
print('Случайный лес регрессия. Прочность при сдвиге:', model_rf_lss.predict(X))
print('XGB регрессия. Прочность при сдвиге:', model_xgb_lss.predict(X))

Линейная регрессия. Прочность при сдвиге: [74.51642475 74.71260408 75.91283845 ... 84.33808221 83.26490418
 83.59986089]


Случайный лес регрессия. Прочность при сдвиге: [78.9 78.9 78.9 ... 86.4 86.4 86.4]
XGB регрессия. Прочность при сдвиге: [78.89904  78.90021  78.90076  ... 86.40007  86.400024 86.39999 ]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


# Модель для предсказания температуры стеклования углепластика Tg

In [55]:
train_data_Tg = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik', 
                         'LSS'], axis=1)

X = np.array(train_data_Tg.drop(['Plastik_Tg'], axis=1))
y = np.array(train_data_Tg.Plastik_Tg.values)

In [56]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_Tg = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_Tg.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [57]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.018
MAPE: 0.005


In [58]:
model_rf_Tg.predict(X)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


array([156., 156., 156., ..., 161., 161., 161.], shape=(6913,))

In [59]:
# save model
with open('autoclave_model_rf_Tg.pkl','wb') as f:
    pickle.dump(model_rf_Tg,f)

In [60]:
model_lr_Tg = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_Tg.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 3.486
MAPE: 0.93


In [61]:
# save model
with open('autoclave_model_lr_Tg.pkl','wb') as f:
    pickle.dump(model_lr_Tg,f)

In [62]:
print('Линейная регрессия. Температура стеклования:', model_lr_Tg.predict(X))
print('Случайный лес регрессия. Температура стеклования:', model_rf_Tg.predict(X))

Линейная регрессия. Температура стеклования: [157.0848237  157.36529173 157.02630514 ... 161.03047571 160.58759076
 160.79798341]
Случайный лес регрессия. Температура стеклования: [156. 156. 156. ... 161. 161. 161.]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished
